# Phase 3 — Type-II charge-pump PLL: operation, limits and fixes (steps 33–45)

The charge pump plus capacitor adds a second integrator: the static phase error disappears, but two integrators give −180° and the loop needs a resistor (a zero) to be stable. This phase derives the CP-PLL completely, designs one for our frequency plan (50 MHz × 32 = 1.6 GHz) — the simulated Type-II baseline of the step-8 decision — and then shows the three limits that push a ring-oscillator design toward the sub-sampling PLL: N² noise, charge-pump spurs, and the bandwidth ceiling.

**How to use this notebook.** Every step follows the same loop: **What happens** (the observation) →
**How it happens** (the mechanism and the equations, in Razavi's notation) → **Why it happens** (the physical
reason) → an interactive figure → **Try it** (what to change and what to look for) → **Interview questions**
(click a question to open the answer; try to answer it aloud first).

Notation (Razavi, *RF Microelectronics* ch. 9 and *Design of CMOS PLLs*): $\phi_{in}$, $\phi_{out}$ input and output
phase; $\omega_{out} = \omega_0 + K_{VCO} V_{cont}$; $K_{PD}$ phase-detector gain; $I_p$ charge-pump current;
$R_1, C_1, C_2$ loop filter; $N$ divide ratio; $\omega_n$, $\zeta$ natural frequency and damping;
$H(s) = \Phi_{out}/\Phi_{in}$. Frequencies $f$ in Hz, angular $\omega = 2\pi f$ in rad/s.
Design numbers are the submitted proposal's: $f_{REF} = 50$ MHz, $N = 32$, $f_{out} = 1.6$ GHz, $f_c \approx 2.5$ MHz.

## Contents

- [Step 33 — The phase-frequency detector](#step-33)
- [Step 34 — Charge pump into a bare capacitor: why it cannot be stable, and why a series R fixes it](#step-34)
- [Step 35 — The second-order Type-II loop: H(s), ω_n, ζ, and the zero's peaking](#step-35)
- [Step 36 — Third order: C2 for ripple, pole placement, phase-margin optimum vs C1/C2](#step-36)
- [Step 37 — Sampled-data reality: Gardner's limit, why f_c ≤ f_REF/10](#step-37)
- [Step 38 — Design procedure with numbers: 50 MHz × 32 = 1.6 GHz, the Type-II baseline](#step-38)
- [Step 39 — Limit: PFD, CP and divider noise × N²](#step-39)
- [Step 40 — Limit: CP mismatch, leakage, charge sharing, clock feedthrough → ripple → reference spurs](#step-40)
- [Step 41 — Limit summary: dead zone, C1 area, and the bandwidth–spur–noise trade-off](#step-41)
- [Step 42 — Fixes: reset delay, matched CP with op-amp equalised drains, dump-node buffer](#step-42)
- [Step 43 — Fixes: sample-reset/S&H filters, calibration, and a PD gain that does not shrink with N](#step-43)
- [Step 44 — Fix for ring-VCO noise: a wider f_c needs lower in-band noise](#step-44)
- [Step 45 — Lab: behavioural Type-II PLL at 1.6 GHz (frequency steps, slewing, settling)](#step-45)

*13 steps, 118 interview questions.*

In [ ]:
# setup - run this cell first (Shift+Enter). Figures are interactive: drag the slider under each plot,
# hover for values, drag to zoom, double-click to reset.
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sspll_learn as sl
from sspll_learn import P, show, sweep, bode, kT
np.set_printoptions(precision=4, suppress=True)
print("ready - proposal design point:", f"{P['fref']/1e6:.0f} MHz x {P['N']} = {P['f0']/1e9:.2f} GHz")

<a id='step-33'></a>
## Step 33 — The phase-frequency detector

*Roadmap step 33: three states, reset path, frequency detection, dead zone*

### What happens
Two edges arrive at the PFD; one output (UP or DN) goes high at the first edge and both reset shortly after the second, so the pulse width equals the time difference.

### How it happens
Two D flip-flops with D = 1 are clocked by REF and DIV; an AND gate detects UP·DN = 1 and resets both after $t_{rst}$. $t_{UP} = \frac{\Delta\phi}{2\pi}T_{REF}$ (linear over ±2π). With unequal frequencies extra edges of the faster input find the PFD already set, so one output stays high most of the time.

### Why it happens
Because the PFD stores *which edge came first*, its average output keeps the sign of a frequency error — that is why a CP-PLL pulls in over the whole VCO range. The dead zone appears because the charge-pump switches need a minimum pulse to turn on; the reset delay is the cure.

### Why it happens
The PFD is a memory element (two flip-flops): it records *which* edge arrived first and holds that state until
the other edge comes. That memory is what lets it tell frequency apart from phase, and the time it holds the state is exactly the
time difference — hence a pulse width proportional to $\Delta\phi$. The dead zone comes from the charge pump, not the logic: real
switches need a minimum pulse to turn on.

### What happens
A PFD has two D flip-flops (D tied high) clocked by REF and DIV, and an AND gate that resets both. A REF edge sets **UP**; a DIV
edge sets **DN**; when both are high the AND resets them after a short delay $t_{rst}$.

### How (three states)
| State | UP | DN | Leaves on |
|---|---|---|---|
| S0 (idle) | 0 | 0 | REF edge → S+, DIV edge → S− |
| S+ (REF leads) | 1 | 0 | DIV edge → reset → S0 |
| S− (DIV leads) | 0 | 1 | REF edge → reset → S0 |

The UP (or DN) pulse width equals the time between the edges: $t_{UP} = \frac{\Delta\phi}{2\pi}T_{REF}$, linear over $\pm2\pi$.

**Frequency detection.** If REF is faster, two REF edges can arrive between DIV edges; the second finds the PFD already in S+, so
UP stays high most of the time and the average output keeps the sign of the frequency error. This gives unlimited pull-in.

**Dead zone.** For very small $\Delta\phi$ the UP pulse is too short for the charge-pump switches to turn on fully; no charge flows,
so the loop gain is zero near lock and the phase wanders inside a band (jitter). **Fix:** the reset delay $t_{rst}$ keeps *both*
UP and DN on for a minimum time every cycle, so even tiny differences produce a proportional net charge.

In [ ]:
def pfd_sim(f_ratio, ncyc=40, phi0=0.3):
    # edges: REF at k/1, DIV at (k + phi0)/f_ratio ; returns time axis and UP-DN state
    tr = np.arange(0, ncyc+1)*1.0
    td = (np.arange(0, int(ncyc*f_ratio)+2) + phi0)/f_ratio
    ev = sorted([(t, "R") for t in tr] + [(t, "D") for t in td])
    t_axis = np.linspace(0, ncyc, 8000); st = np.zeros_like(t_axis)
    s, last = 0, 0.0
    times, states = [0.0], [0]
    for t, e in ev:
        if e == "R": s = 0 if s == -1 else 1
        else:        s = 0 if s == 1 else -1
        times.append(t); states.append(s)
    idx = np.searchsorted(times, t_axis, side="right") - 1
    return t_axis, np.array(states)[idx]
def frame(r):
    t, s = pfd_sim(r)
    return ([dict(x=t, y=s, name="UP - DN (PFD state)", line=dict(shape="hv")),
             dict(x=t, y=np.cumsum(s)/(np.arange(len(s))+1), name="running average")],
            f"f_REF / f_DIV = {r}: average output {s.mean():+.2f} -> " + ("equal frequencies: the average is just the 0.3-cycle phase error"
            if r == 1.0 else "pushes VCO up" if s.mean() > 0 else "pushes VCO down"))
fig = sweep([0.8, 0.9, 0.95, 1.0, 1.05, 1.1, 1.25], frame, "f_REF / f_DIV", active=4,
            layout=lambda f: f.update_xaxes(title="time (reference periods)").update_yaxes(range=[-1.2, 1.2]))
show(fig, 420)

In [ ]:
# Dead zone: average CP current vs phase error when pulses shorter than t_on produce no current
dphi = np.linspace(-0.2, 0.2, 801)                                # rad
T = 20e-9; ton = 150e-12                                           # charge-pump turn-on time
def avg_current(dp, trst):
    w = abs(dp)/(2*np.pi)*T                                         # PFD pulse width
    if trst == 0:
        eff = np.maximum(w - ton, 0)                                # pulse must exceed t_on
    else:
        eff = np.where(trst > ton, w, np.maximum(w + trst - ton, 0) - np.maximum(trst - ton, 0))
    return np.sign(dp)*eff/T
fig = go.Figure()
for trst, nm in ((0, "no reset delay: dead zone"), (100e-12, "t_rst = 100 ps (< t_on)"), (300e-12, "t_rst = 300 ps (> t_on): linear")):
    fig.add_trace(go.Scatter(x=np.degrees(dphi), y=avg_current(dphi, trst), name=nm))
fig.update_xaxes(title="phase error (deg)").update_yaxes(title="average current / I_p")
show(fig, 400)

### Interview questions — step 33 (12)

<details><summary><b>Q1.</b> Draw and explain the PFD state diagram.</summary>

Three states: idle (UP = DN = 0), UP (REF edge came first), DN (DIV edge came first). The opposite edge moves it back to idle through the reset path.

</details>

<details><summary><b>Q2.</b> Why does the PFD detect frequency?</summary>

When one input is faster, it delivers more edges; the PFD stays in that input's state most of the time, so the average output has the sign of the frequency difference.

</details>

<details><summary><b>Q3.</b> What is the linear range of a PFD?</summary>

±2π (±one reference period). Beyond that it wraps and the characteristic becomes periodic with a sawtooth shape (but keeps the right sign on average in frequency mode).

</details>

<details><summary><b>Q4.</b> What is the dead zone and why does it matter?</summary>

A range of small phase errors where the CP produces no net charge because the pulses are too short to turn the switches on. The loop gain is zero there, so phase wanders randomly — extra jitter.

</details>

<details><summary><b>Q5.</b> How does a reset delay remove the dead zone?</summary>

Both UP and DN stay on for at least $t_{rst}$ each cycle, so the switches are always fully on; the net charge is $I_p\times$ (difference of pulse widths), linear down to zero.

</details>

<details><summary><b>Q6.</b> What does the reset delay cost?</summary>

Both currents flow every cycle: any UP/DN mismatch now creates a net charge pulse every period → static phase offset and reference spurs; also more CP noise injected.

</details>

<details><summary><b>Q7.</b> What limits the maximum operating frequency of a PFD?</summary>

The reset path delay: an edge arriving during reset is missed. The PFD fails above roughly $1/(2t_{reset})$ and its range shrinks to $2\pi(1 - t_{rst}/T)$.

</details>

<details><summary><b>Q8.</b> Why is the PFD input the divided VCO and not the VCO itself?</summary>

The flip-flops and charge pump cannot run at the VCO frequency, and the comparison must be at $f_{REF}$; hence the divider and its N² penalty.

</details>

<details><summary><b>Q9.</b> What is a 'missing edge' and when does it occur?</summary>

During acquisition, two edges of one input between edges of the other; the PFD ignores the extra edge. That is exactly what gives it frequency detection.

</details>

<details><summary><b>Q10.</b> How is a PFD different from a bang-bang PD?</summary>

A PFD gives a pulse width proportional to the phase error (linear); a bang-bang PD gives only its sign.

</details>

<details><summary><b>Q11.</b> In our SSPLL, where is a PFD used?</summary>

Only in the FLL: REF vs the /32 divider output, with a dead zone added on purpose so it is silent once the phase error is small.

</details>

<details><summary><b>Q12.</b> Why would we *want* a dead zone in the FLL?</summary>

So the FLL stops injecting noise and spurs after frequency acquisition; inside the dead zone only the SSPD loop acts.

</details>

<a id='step-34'></a>
## Step 34 — Charge pump into a bare capacitor: why it cannot be stable, and why a series R fixes it

*Roadmap step 34: Bode plots*

### What happens
PFD + CP + a single capacitor $C_1$ + VCO: the loop locks at first glance but rings forever.

### How
$$G(s) = \frac{I_p}{2\pi}\cdot\frac{1}{C_1s}\cdot\frac{K_{VCO}}{Ns} = \frac{I_pK_{VCO}}{2\pi NC_1}\cdot\frac{1}{s^2}\quad\Rightarrow\quad
\angle G = -180°\ \text{at all }\omega.$$
Closed loop: $H \propto \frac{1}{s^2+\omega_n^2}$, poles at $\pm j\omega_n$ — an undamped oscillator (ζ = 0).

Adding $R_1$ in series with $C_1$: $Z = R_1 + \frac{1}{C_1 s}$, so
$$G(s) = \frac{I_pK_{VCO}}{2\pi NC_1}\cdot\frac{1+sR_1C_1}{s^2}:$$
a zero at $\omega_z = 1/R_1C_1$ adds up to +90°.

### Why physically
With only $C_1$, the control voltage is the *integral* of past phase error: by the time the phase error reaches zero, $V_{cont}$
has overshot. The resistor adds a term *proportional* to the present phase error ($I_pR_1$ during each pulse) — an immediate
correction, like the "P" in a PI controller, which damps the loop.

In [ ]:
from scipy import signal
Ip, Kv, N = 100e-6, 2*np.pi*300e6, 32
d = sl.loop_design_type2(2.5e6, 60, Ip, Kv, N); C1 = d['C1'] + d['C2']
f = np.logspace(4, 9, 400); s = 2j*np.pi*f; t = np.linspace(0, 4e-6, 2000)
def frame(Rk):
    R = Rk*1e3
    G = Ip*Kv/(2*np.pi*N*C1)*(1+s*R*C1)/s**2
    k = Ip*Kv/(2*np.pi*N*C1)
    _, y = signal.step(signal.lti([k*R*C1, k], [1, k*R*C1, k]), T=t)
    pm = 180 + np.degrees(np.angle(G[np.argmin(abs(abs(G)-1))]))
    return ([dict(x=f, y=20*np.log10(abs(G)), name="|G| (dB)", row=1, col=1),
             dict(x=f, y=np.degrees(np.angle(G)), name="phase G (deg)", row=1, col=2),
             dict(x=t*1e6, y=y, name="phase-step response H/N", row=2, col=1)],
            f"R1 = {Rk} kOhm: phase margin {pm:.0f} deg" + ("  (C only: undamped oscillation)" if Rk == 0 else ""))
fig = sweep([0, 1, 2, 5, 10, 18, 40], frame, "R1 (kOhm)", active=5, rows=2, cols=2,
            subplot_titles=("|G|", "phase of G", "step response", ""), height=620,
            layout=lambda f: f.update_xaxes(type="log", row=1, col=1).update_xaxes(type="log", row=1, col=2)
                              .update_yaxes(range=[-185, -80], row=1, col=2).update_yaxes(range=[0, 2.1], row=2, col=1)
                              .update_xaxes(title_text="time (us)", row=2, col=1))
show(fig)

### Interview questions — step 34 (9)

<details><summary><b>Q1.</b> Why is a CP-PLL with only a capacitor unstable?</summary>

Two integrators: $G\propto1/s^2$, phase −180° everywhere, phase margin 0; closed-loop poles on the imaginary axis.

</details>

<details><summary><b>Q2.</b> What does R1 add in the s-domain and physically?</summary>

A zero at $1/R_1C_1$ (+90° phase). Physically, a proportional path: $V_{cont}$ jumps by $I_pR_1$ during each pulse, correcting in proportion to the present error.

</details>

<details><summary><b>Q3.</b> What is the PI-controller analogy?</summary>

$Z = R_1 + 1/(sC_1)$ is proportional (R1) plus integral (C1) control of the phase error.

</details>

<details><summary><b>Q4.</b> What is the side effect of R1?</summary>

A voltage step $I_pR_1$ on $V_{cont}$ every time the CP turns on → ripple → spurs; hence C2.

</details>

<details><summary><b>Q5.</b> If R1 = 0, what frequency does the loop ring at?</summary>

$\omega_n = \sqrt{I_pK_{VCO}/(2\pi NC_1)}$.

</details>

<details><summary><b>Q6.</b> Can you stabilise the loop with a zero elsewhere?</summary>

Yes: e.g. a feedforward proportional path (a second CP into a resistor, or a dual-path filter) creates the zero without a series resistor; used to shrink C1.

</details>

<details><summary><b>Q7.</b> Does R1 change the steady-state phase error?</summary>

No: the integrator (C1) still guarantees zero error for a frequency step.

</details>

<details><summary><b>Q8.</b> What happens to phase margin as R1 grows very large?</summary>

In the 2nd-order model it approaches 90°; with C2 present, crossover moves toward the third pole and PM drops again.

</details>

<details><summary><b>Q9.</b> How does R1 thermal noise appear at the output?</summary>

As a voltage on $V_{cont}$, converted by $K_{VCO}$ and shaped like VCO noise (high-pass by the loop); larger R1 means more noise.

</details>

<a id='step-35'></a>
## Step 35 — The second-order Type-II loop: H(s), ω_n, ζ, and the zero's peaking

*Roadmap step 35: derivation and plots*

### What happens
With a series R1 the CP-PLL becomes a stable second-order loop whose closed-loop response has a zero, so it peaks (|H| > 1) near ω_n and overshoots in the step response even when ζ ≥ 1.

### How it happens
Write $G = \frac{I_pK_{VCO}}{2\pi NC_1}\frac{1+sR_1C_1}{s^2}$, close the loop $H/N = G/(1+G)$ and match the denominator to $s^2+2\zeta\omega_ns+\omega_n^2$: $\omega_n^2 = I_pK_{VCO}/(2\pi NC_1)$, $2\zeta\omega_n = I_pK_{VCO}R_1/(2\pi N)$. The numerator $2\zeta\omega_ns+\omega_n^2$ is the closed-loop zero.

### Why it happens
The zero comes from R1's proportional path: it adds phase (stability) but also lets the input pass with gain above 1 near ω_n. Both ω_n and ζ grow with √(I_p K_VCO), so PVT spread of K_VCO moves bandwidth and damping together.

### What happens
With $R_1$ in series with $C_1$ the charge-pump PLL becomes a stable second-order Type-II loop: it settles after a
phase step with an overshoot set by ζ, and its closed-loop response shows a small peak near $\omega_n$ even when well damped.

### How (Razavi, *RF Microelectronics* §9.3; divider included)
$$G(s) = \frac{I_pK_{VCO}}{2\pi NC_1}\frac{1+sR_1C_1}{s^2},\qquad
\frac{H(s)}{N} = \frac{G}{1+G} = \frac{2\zeta\omega_n s + \omega_n^2}{s^2 + 2\zeta\omega_n s + \omega_n^2}$$
$$\boxed{\omega_n = \sqrt{\frac{I_pK_{VCO}}{2\pi NC_1}},\qquad \zeta = \frac{R_1}{2}\sqrt{\frac{I_pC_1K_{VCO}}{2\pi N}}}$$
The zero of $H$ at $\omega_n/(2\zeta)$ makes $|H/N| > 1$ near $\omega_n$ (**peaking**) for any ζ, unlike a pure second-order
low-pass. For ζ ≥ 1 the poles are real; the −3 dB bandwidth is about $2\zeta\omega_n$ for large ζ.

### Why it matters
ζ grows with $R_1$ and with $\sqrt{I_pK_{VCO}}$; bandwidth grows with $\sqrt{I_pK_{VCO}/C_1}$. Since $K_{VCO}$ varies with PVT, both
move together — designs check the extremes. Peaking amplifies in-band noise and is the reason "too much" bandwidth hurts.

In [ ]:
from scipy import signal
f = np.logspace(4, 9, 500); t = np.linspace(0, 3e-6, 1500); wn = 2*np.pi*1e6
def frame(z):
    num, den = [2*z*wn, wn**2], [1, 2*z*wn, wn**2]
    w, mag, _ = signal.bode(signal.lti(num, den), w=2*np.pi*f)
    _, y = signal.step(signal.lti(num, den), T=t)
    return ([dict(x=f, y=mag, name="|H/N| (dB)", row=1, col=1),
             dict(x=t*1e6, y=y, name="phase-step response", row=1, col=2)],
            f"zeta = {z}: peaking {mag.max():.2f} dB, overshoot {100*(y.max()-1):.0f} %, f_-3dB = {f[np.where(mag > -3)[0][-1]]/1e6:.2f} MHz (f_n = 1 MHz)")
fig = sweep([0.3, 0.5, 0.707, 1.0, 1.5, 2.0, 3.0], frame, "zeta", active=2, rows=1, cols=2,
            subplot_titles=("closed-loop magnitude", "step response"), height=420,
            layout=lambda f: f.update_xaxes(type="log", title_text="Hz", row=1, col=1).update_yaxes(range=[-30, 8], row=1, col=1)
                              .update_xaxes(title_text="time (us)", row=1, col=2))
show(fig)

### Interview questions — step 35 (10)

<details><summary><b>Q1.</b> Derive ω_n and ζ of the second-order CP-PLL.</summary>

From $1+G = 0$: $s^2 + \frac{I_pK_{VCO}R_1}{2\pi N}s + \frac{I_pK_{VCO}}{2\pi NC_1} = 0$ → $\omega_n^2 = I_pK_{VCO}/(2\pi NC_1)$, $2\zeta\omega_n = I_pK_{VCO}R_1/(2\pi N)$ → $\zeta = \frac{R_1}{2}\sqrt{I_pC_1K_{VCO}/(2\pi N)}$.

</details>

<details><summary><b>Q2.</b> Why does a Type-II loop show peaking even with ζ = 1?</summary>

The closed-loop zero at $\omega_n/2\zeta$ lifts the response before the poles roll it off; ζ = 1 still peaks ~1.25 dB.

</details>

<details><summary><b>Q3.</b> How does ζ depend on I_p and K_VCO?</summary>

$\zeta\propto\sqrt{I_pK_{VCO}}$: a 2× spread in $K_{VCO}$ changes ζ by 1.41× and ω_n by 1.41×.

</details>

<details><summary><b>Q4.</b> What is the bandwidth for large ζ?</summary>

Approximately $2\zeta\omega_n = I_pK_{VCO}R_1/(2\pi N)$ — set by R1, the CP and the VCO, independent of C1.

</details>

<details><summary><b>Q5.</b> What is the settling time to 1% for a second-order loop?</summary>

Approximately $\ln(100)/(\zeta\omega_n) = 4.6/(\zeta\omega_n)$ for ζ < 1.

</details>

<details><summary><b>Q6.</b> Why do CP-PLL designers often use ζ ≈ 0.7–1?</summary>

Compromise between peaking/overshoot and settling speed; noise peaking and lock time are both reasonable.

</details>

<details><summary><b>Q7.</b> What changes if N doubles with the same filter?</summary>

ω_n falls by $\sqrt2$, ζ falls by $\sqrt2$, in-band noise rises 6 dB.

</details>

<details><summary><b>Q8.</b> Write H(s) in terms of ω_n and ζ.</summary>

$H/N = (2\zeta\omega_ns + \omega_n^2)/(s^2 + 2\zeta\omega_ns + \omega_n^2)$.

</details>

<details><summary><b>Q9.</b> Is the second-order model good enough for design?</summary>

For a first estimate; real loops add C2 (third pole), parasitic poles and the sampling delay, so final PM is checked with the third-order and sampled-data models.

</details>

<details><summary><b>Q10.</b> What does the closed-loop zero mean for jitter transfer?</summary>

Jitter (and noise) from the reference is amplified by the peaking near ω_n; important in cascaded PLLs and CDR jitter-transfer specs.

</details>

<a id='step-36'></a>
## Step 36 — Third order: C2 for ripple, pole placement, phase-margin optimum vs C1/C2

*Roadmap step 36: design curves*

### What happens
$C_2$ from $V_{cont}$ to ground smooths the $I_pR_1$ steps. It adds a pole $\omega_{p3} = \frac{C_1+C_2}{R_1C_1C_2}$.

### How
With $b^2 = \omega_{p3}/\omega_z = 1 + C_1/C_2$ and crossover at the geometric mean,
$$PM_{max} = \arctan b - \arctan\frac1b = \arctan\frac{b^2-1}{2b}.$$
And the crossover condition gives the closed-form design (used throughout this project, `sl.loop_design_type2`):
$$C_{tot} = C_1+C_2 = \frac{b\,I_pK_{VCO}}{2\pi N\omega_c^2},\quad C_2 = \frac{C_{tot}}{b^2},\quad C_1 = C_{tot}-C_2,\quad R_1 = \frac{b}{\omega_cC_1}.$$

### Why there is a trade-off
Bigger $C_2$ (smaller $C_1/C_2$) → better ripple filtering (lower spurs) but the pole approaches crossover → less PM. Smaller $C_2$ →
more PM but the third pole moves up and filters the ripple less.

In [ ]:
ratio = np.logspace(0, 2.5, 200)
bb = np.sqrt(1+ratio)
pm = np.degrees(np.arctan(bb) - np.arctan(1/bb))
fig = go.Figure(go.Scatter(x=ratio, y=pm, name="max phase margin"))
for r, lab in ((4.83, "45 deg"), (12.9, "60 deg"), (32.2, "70 deg")):
    fig.add_trace(go.Scatter(x=[r], y=[np.degrees(np.arctan(np.sqrt(1+r)) - np.arctan(1/np.sqrt(1+r)))], mode="markers+text",
                             text=[f"C1/C2 = {r}"], textposition="top left", marker=dict(size=10), name=lab))
fig.update_xaxes(type="log", title="C1 / C2").update_yaxes(title="maximum phase margin (deg)")
show(fig, 400)

In [ ]:
Ip, Kv, N = 100e-6, 2*np.pi*300e6, 32
f = np.logspace(4, 9, 400); s = 2j*np.pi*f
def frame(pm_t):
    d = sl.loop_design_type2(2.5e6, pm_t, Ip, Kv, N)
    Z = (1+s*d['R']*d['C1'])/(s*(d['C1']+d['C2'])*(1+s*d['R']*d['C1']*d['C2']/(d['C1']+d['C2'])))
    G = Ip/(2*np.pi)*Z*Kv/(N*s)
    step = Ip*d['R']*d['C1']/(d['C1']+d['C2'])   # V_cont jump at the start of a CP pulse (charge splits)
    return ([dict(x=f, y=20*np.log10(abs(G)), name="|G| (dB)", row=1, col=1),
             dict(x=f, y=np.degrees(np.unwrap(np.angle(G))), name="phase G", row=1, col=2)],
            f"PM {pm_t} deg at f_c = 2.5 MHz: C1/C2 = {d['C1']/d['C2']:.1f}, C1 = {d['C1']*1e12:.1f} pF, C2 = {d['C2']*1e12:.2f} pF, "
            f"R1 = {d['R']/1e3:.1f} kOhm, zero {d['fz']/1e6:.2f} MHz, pole {d['fp']/1e6:.1f} MHz")
fig = sweep([40, 45, 50, 55, 60, 65, 70], frame, "target phase margin (deg)", active=4, rows=1, cols=2,
            subplot_titles=("|G|", "phase of G"), height=420,
            layout=lambda f: f.update_xaxes(type="log", title_text="Hz").update_yaxes(range=[-185, -90], row=1, col=2))
show(fig)

### Interview questions — step 36 (10)

<details><summary><b>Q1.</b> Why is C2 needed?</summary>

To filter the voltage step $I_pR_1$ (and switching glitches) on $V_{cont}$ that would otherwise modulate the VCO every reference period.

</details>

<details><summary><b>Q2.</b> Where is the third pole?</summary>

$\omega_{p3} = (C_1+C_2)/(R_1C_1C_2) = 1/(R_1C_s)$.

</details>

<details><summary><b>Q3.</b> Derive the maximum PM for a given C1/C2.</summary>

PM peaks at $\sqrt{\omega_z\omega_p}$: $\arctan b - \arctan(1/b)$ with $b^2 = 1 + C_1/C_2$.

</details>

<details><summary><b>Q4.</b> Typical C1/C2 for 60° PM?</summary>

≈ 12.9 ($b = 3.73$).

</details>

<details><summary><b>Q5.</b> Give the closed-form loop-filter design for given f_c, PM, I_p, K_VCO, N.</summary>

$b = \tan(45°+PM/2)$, $C_{tot} = bI_pK_{VCO}/(2\pi N\omega_c^2)$, $C_2 = C_{tot}/b^2$, $C_1 = C_{tot} - C_2$, $R_1 = b/(\omega_cC_1)$.

</details>

<details><summary><b>Q6.</b> How does C_tot scale with bandwidth?</summary>

$\propto 1/\omega_c^2$: a 2× wider loop needs 4× less capacitance.

</details>

<details><summary><b>Q7.</b> What is a 4th-order (two extra poles) loop filter for?</summary>

An additional RC after the third-order filter for more spur rejection; it costs a few degrees of PM if placed ~5–10× above crossover.

</details>

<details><summary><b>Q8.</b> Why does a large C2 reduce phase margin?</summary>

It lowers the third pole toward crossover; the pole's phase lag adds at $\omega_c$.

</details>

<details><summary><b>Q9.</b> Why place zero and pole symmetrically about crossover?</summary>

The phase bump $\arctan(\omega/\omega_z) - \arctan(\omega/\omega_p)$ is maximum at their geometric mean; crossover there gives the maximum PM for that C1/C2.

</details>

<details><summary><b>Q10.</b> What happens to the design if K_VCO is 2× higher than assumed?</summary>

Crossover rises ~1.6–2×, no longer at the geometric mean; PM drops by ~5–10° and spurs/peaking increase — design with margin or calibrate the CP current.

</details>

<a id='step-37'></a>
## Step 37 — Sampled-data reality: Gardner's limit, why f_c ≤ f_REF/10

*Roadmap step 37: plot*

### What happens
The PFD/CP acts once per reference period. The continuous-time model assumes a smooth current, which is only true when the loop
changes little per period. As $f_c$ approaches $f_{REF}$, the real loop is less stable than predicted and finally oscillates, even if
the CT phase margin is fine. Gardner (1980) derived the stability boundary of the sampled CP-PLL; the practical result is the
rule **$f_c \lesssim f_{REF}/10$**.

### How (an intuitive model)
Updating once per period is like a zero-order hold, whose phase is a delay of about $T_{REF}/2$:
$$\Delta PM \approx 360°\cdot f_c\cdot\frac{T_{REF}}{2} = 180°\,\frac{f_c}{f_{REF}}.$$
At $f_{REF}/10$ this is 18° (Thijssen et al. quote the same 18° for a sample-and-hold SSPD at $f_{REF}/10$); at $f_{REF}/4$, 45°.

### Why it matters for us
Our SSPLL has an explicit hold (sample, then hold until the next sample), so the delay is real, not just a modelling artefact.
Nagam & Kinget use $\omega_{REF}/20$; the proposal's 2.5 MHz is exactly $f_{REF}/20$ (9° delay penalty).

In [ ]:
Ip, Kv, N, fref = 100e-6, 2*np.pi*300e6, 32, 50e6
ratios = np.array([1/40, 1/30, 1/20, 1/15, 1/10, 1/8, 1/6, 1/5, 1/4, 1/3.5, 1/3])
over = []
for r in ratios:
    d = sl.loop_design_type2(fref*r, 60, Ip, Kv, N)
    t, e, _ = sl.cp_pll_sim(Ip, d['R'], d['C1'], d['C2'], Kv, N, fref, ncyc=int(30/r), phi_step=0.2)
    over.append(max(-e.min()/0.2, 0))
fig = make_subplots(rows=1, cols=2, subplot_titles=("effective phase margin (CT 60 deg minus hold delay)", "sampled-data simulation: undershoot of the error"))
fig.add_trace(go.Scatter(x=1/ratios, y=60 - 180*ratios, mode="lines+markers", name="60 - 180 f_c/f_REF"), 1, 1)
fig.add_trace(go.Scatter(x=1/ratios, y=np.array(over)*100, mode="lines+markers", name="undershoot (%)"), 1, 2)
fig.update_xaxes(type="log", title_text="f_REF / f_c").update_yaxes(title_text="deg", row=1, col=1).update_yaxes(title_text="%", type="log", row=1, col=2)
show(fig, 420)

### Interview questions — step 37 (9)

<details><summary><b>Q1.</b> Why can't a CP-PLL bandwidth approach f_REF?</summary>

The detector updates once per period; the loop sees a delay of ~T/2 and acts on stale information. Beyond ~$f_{REF}/10$ the sampled loop loses margin quickly and becomes unstable (Gardner's limit).

</details>

<details><summary><b>Q2.</b> Estimate the phase-margin penalty of a half-period delay at f_c = f_REF/10.</summary>

$180°\times0.1 = 18°$.

</details>

<details><summary><b>Q3.</b> Is the continuous-time approximation pessimistic or optimistic?</summary>

Optimistic about stability at high $f_c/f_{REF}$: it ignores the sampling delay.

</details>

<details><summary><b>Q4.</b> How do you analyse a CP-PLL exactly?</summary>

As a discrete-time system: z-domain model of the PFD/CP charge per cycle and the exact filter/VCO evolution between edges (impulse-invariant), or a cycle-by-cycle simulation like `sl.cp_pll_sim`.

</details>

<details><summary><b>Q5.</b> Why do SSPLLs typically use f_REF/20?</summary>

The hold adds a real half-period delay on top of sampling; spur filtering and Nagam's design practice also favour ~$f_{REF}/20$.

</details>

<details><summary><b>Q6.</b> How does a pulser reduce the hold delay of an SSPD?</summary>

The charge pump is enabled only during a short pulse right after sampling; the effective delay becomes ~ the pulse position, shorter than T/2 (Thijssen notes this improves stability).

</details>

<details><summary><b>Q7.</b> What does Gardner's analysis predict for very large loop gain?</summary>

Oscillation at ~$f_{REF}/2$: the sampled loop's poles leave the unit circle.

</details>

<details><summary><b>Q8.</b> What is the effect of the divider on the sampling picture?</summary>

The comparison rate is $f_{REF}$ regardless of N; N only scales the gain.

</details>

<details><summary><b>Q9.</b> Name two ways to get more bandwidth than f_REF/10 allows.</summary>

Raise $f_{REF}$ (higher-frequency crystal or reference doubler), or use both edges / multiple phase detectors per period.

</details>

<a id='step-38'></a>
## Step 38 — Design procedure with numbers: 50 MHz × 32 = 1.6 GHz, the Type-II baseline

*Roadmap step 38: pick f_c and PM, get R1, C1, C2, I_p, check PDK legality*

### What happens
We turn the theory of steps 34–37 into real component values for the simulated Type-II baseline: f_REF = 50 MHz, N = 32, f_c = 2.5 MHz, PM = 60°.

### How it happens
Pick f_c and PM → b = tan(45° + PM/2) → C_1/C_2 = b² − 1 → crossover condition gives C_tot = b·I_p·K_VCO/(2πNω_c²) → C_2 = C_tot/b², C_1 = C_tot − C_2, R_1 = b/(ω_cC_1) → convert to SKY130 geometry (xhigh-poly length, MIM/MOS-cap area).

### Why it happens
A design is only real when every element is legal in the PDK and survives the R/C corners; doing the arithmetic once by hand shows which knob (I_p) trades area against noise, and that the ripple step I_pR_1 is fixed by the loop itself.

### What happens
We turn the equations of steps 35–36 into actual component values for the Type-II baseline at our frequency
plan (50 MHz × 32 = 1.6 GHz, $f_c$ = 2.5 MHz, PM = 60°) and check that every component can be drawn in SKY130.

### How it happens
Fix $f_c$, PM, $N$, $K_{VCO}$ and choose $I_p$; then $b=\tan(45°+PM/2)$, $C_{tot}=bI_pK_{VCO}/(2\pi N\omega_c^2)$,
$C_2=C_{tot}/b^2$, $C_1=C_{tot}-C_2$, $R_1=b/(\omega_cC_1)$; finally convert R to a poly-resistor length and C to MIM or MOS-cap
area with the cheat-sheet densities.

### Procedure
1. **Bandwidth.** $f_c = f_{REF}/20 = 2.5$ MHz (same as the SSPLL, for a fair comparison; step 8).
2. **Phase margin.** 60° → $b = 3.73$, $C_1/C_2 = 12.9$.
3. **Gains.** $N = 32$; $K_{VCO} = 2\pi\cdot300$ MHz/V (**assumed** fine-tuning gain until Phase 7 measures the ring); $I_p$ free.
4. **Filter** (step 36 closed form). 5. **PDK check** against the cheat sheet: xhigh-poly resistor (≈5.71 kΩ/µm at 0.35 µm width
plus ≈0.59 kΩ heads, from `correlation/corr_rc.sp`), MIM ≈2.03 fF/µm² (30×30 µm max tile), 1.8 V MOS cap ≈7.48 fF/µm² (only above ~0.6 V).

### Why $I_p$ is the main knob
$C_{tot}\propto I_p$ and $R_1\propto 1/I_p$. A large $I_p$ gives big capacitors (area, power) but a small $R_1$ (less thermal noise)
and makes CP noise, leakage and mismatch *relatively* smaller. A small $I_p$ gives a compact filter but a large, noisy $R_1$.
One quantity does **not** change with $I_p$: the ripple step $I_pR_1 = b\,I_p/(\omega_cC_1)$, fixed by $f_c$, PM, $K_{VCO}$ and $N$.

In [ ]:
Kv, N, fc, pmt = 2*np.pi*300e6, 32, 2.5e6, 60
def frame(Ip_uA):
    d = sl.loop_design_type2(fc, pmt, Ip_uA*1e-6, Kv, N)
    L_r = max((d['R'] - 590)/5714, 0.5)                    # xhigh_po_0p35 length (um), >= 0.5 um
    A_mim = d['C1']/2.03e-15; A_mos = d['C1']/7.48e-15     # um^2
    A_c2 = d['C2']/2.03e-15
    names = ["C1 as MIM", "C1 as MOS cap", "C2 as MIM", "R1 (0.35 um xhigh)"]
    areas = [A_mim, A_mos, A_c2, 0.35*L_r + 2*0.35*2]
    return ([dict(x=names, y=areas, mode="markers+text", text=[f"{a:.0f}" for a in areas], textposition="top center",
                  marker=dict(size=14), name="area (um^2)")],
            f"I_p = {Ip_uA} uA: R1 = {d['R']/1e3:.2f} kOhm (L = {L_r:.1f} um), C1 = {d['C1']*1e12:.2f} pF, C2 = {d['C2']*1e12:.3f} pF, "
            f"I_p R1 = {Ip_uA*1e-6*d['R']*1e3:.0f} mV")
fig = sweep([10, 25, 50, 100, 200, 400], frame, "charge-pump current I_p (uA)", active=3,
            layout=lambda f: f.update_yaxes(type="log", title="area (um^2)", range=[0, 5.3]))
show(fig, 440)
d = sl.loop_design_type2(fc, pmt, 100e-6, Kv, N)
print({k: f"{v:.4g}" for k, v in d.items()})

**Result for $I_p$ = 100 µA:** $R_1$ = 18.05 kΩ (xhigh 0.35 × ≈3.1 µm), $C_1$ = 13.16 pF (MIM ≈ 6500 µm² = 8 tiles of 30×30 µm,
or MOS cap ≈ 1760 µm²), $C_2$ = 1.02 pF (MIM ≈ 22×22 µm). All legal. This is the Type-II reference design used in the later steps;
the ngspice phase-domain netlist of Phase 1 uses exactly these values. **R/C corners** (ll/hh, step 7) move $R_1C_1$ by −27…+35%:
check PM at those corners before trusting 60°.

### Interview questions — step 38 (10)

<details><summary><b>Q1.</b> Walk me through a CP-PLL loop-filter design.</summary>

Choose $f_c$ (≤ $f_{REF}/10$, here $f_{REF}/20$) and PM (50–60°); from $b = \tan(45°+PM/2)$ get $C_1/C_2$; from the crossover condition get $C_{tot}$ for a chosen $I_p$; then $C_2$, $C_1$ and $R_1 = b/(\omega_cC_1)$; check noise, area, spurs and PVT.

</details>

<details><summary><b>Q2.</b> How do you choose I_p?</summary>

Trade area ($C\propto I_p$) against noise and mismatch: larger $I_p$ lowers R1 (less thermal noise) and makes CP noise/leakage relatively smaller, at the cost of capacitor area and CP power.

</details>

<details><summary><b>Q3.</b> Why is I_p·R1 independent of I_p?</summary>

$R_1 = b/(\omega_cC_1)$ and $C_1\propto I_p$, so $I_pR_1$ depends only on $f_c$, PM, $K_{VCO}$ and N. The ripple step is set by the loop dynamics, not by the CP current.

</details>

<details><summary><b>Q4.</b> How large is 13 pF in SKY130?</summary>

≈6500 µm² of MIM (8 tiles of 30×30) or ≈1760 µm² of 1.8 V MOS cap if biased above ~0.6 V.

</details>

<details><summary><b>Q5.</b> Why use a MOS cap for C1 but MIM for C2?</summary>

C1 is large and only needs to be roughly linear (it sits at $V_{cont}$, well above threshold); C2 is small and should be linear and low-leakage; MIM fits. MOS caps also leak more.

</details>

<details><summary><b>Q6.</b> What is the resistor length for 18 kΩ in xhigh poly?</summary>

≈ (18.05 − 0.59 kΩ heads)/5.71 kΩ/µm ≈ 3.1 µm at 0.35 µm width — legal (L ≥ 0.5 µm).

</details>

<details><summary><b>Q7.</b> How does K_VCO uncertainty enter the design?</summary>

Directly in $C_{tot}$ and $\omega_c$; designs use the nominal value for sizing and check PM and spurs at $K_{VCO,min/max}$.

</details>

<details><summary><b>Q8.</b> Why did we choose the same f_c for the Type-II baseline as for the SSPLL?</summary>

To compare architectures at equal bandwidth: differences in jitter then come from detector noise (N²), not from bandwidth.

</details>

<details><summary><b>Q9.</b> What changes in the design for N = 48 at 49.152 MHz (the old design)?</summary>

N enters $C_{tot}\propto1/N$: 1.5× less capacitance for the same $f_c$ and $I_p$; in-band noise +3.5 dB.

</details>

<details><summary><b>Q10.</b> Which numbers will Phase 7 replace in this design?</summary>

$K_{VCO}$ (ring fine-tune gain over PVT), the real $I_p$ and its mismatch, R/C values at corners, parasitic poles.

</details>

<a id='step-39'></a>
## Step 39 — Limit: PFD, CP and divider noise × N²

*Roadmap step 39: numbers for 50 MHz × 32: +30.1 dB*

### What happens
Inside the loop bandwidth, everything that appears at the PFD input (reference, PFD/CP, divider) reaches the output multiplied by
$N$ in phase: **+20 log N = +30.1 dB** for $N = 32$.

### How (normalised figure of merit)
CP-PLL in-band noise is captured by a normalised floor $\mathcal L_{1Hz}$ (Banerjee):
$$\mathcal L_{in\text{-}band} = \mathcal L_{1Hz} + 10\log f_{REF} + 20\log N.$$
For a fixed output frequency $f_0 = Nf_{REF}$, this is $\mathcal L_{1Hz} + 10\log f_0 + 10\log N$: at a fixed $f_0$, a lower reference
(larger N) costs **+10 log N**. With $\mathcal L_{1Hz} = -218$ dBc/Hz: $-218 + 77 + 30.1 = -110.9$ dBc/Hz at 1.6 GHz.

The divider adds its own jitter (each flip-flop's) and power (the first stages run at GHz).

### Why it limits a ring PLL
A ring needs a wide loop (step 23), but a wide loop lets the ×N² floor through. The SSPLL's detector noise is not multiplied by N²
(−130 dBc/Hz output-referred in our budget), which moves the optimum to wider bandwidth.

In [ ]:
import sys; sys.path.insert(0, "../models")
import pllmodel as pm
Ns = [8, 16, 24, 32, 48, 64]
def frame(Nv):
    fref = 1.6e9/Nv; fc = min(2.5e6, fref/20)
    t2 = pm.TYPE2_TYP.but(N=Nv, fref=fref, fc=fc); ss = pm.TYP.but(N=Nv, fref=fref, fc=fc)
    j2, f, L2 = pm.jitter(t2, "type2"); js, _, Ls = pm.jitter(ss, "sspll")
    floor = -218 + 10*np.log10(fref) + 20*np.log10(Nv)
    return ([dict(x=f, y=L2, name=f"Type-II: {j2['TOTAL']*1e12:.2f} ps"), dict(x=f, y=Ls, name=f"SSPLL: {js['TOTAL']*1e12:.2f} ps")],
            f"N = {Nv}, f_REF = {fref/1e6:.1f} MHz, f_c = {fc/1e6:.2f} MHz: CP-PLL in-band floor {floor:.1f} dBc/Hz (+{20*np.log10(Nv):.1f} dB from N)")
fig = sweep(Ns, frame, "N (f_out fixed at 1.6 GHz)", active=3,
            layout=lambda f: f.update_xaxes(type="log", title="offset (Hz)", range=[3, 8]).update_yaxes(title="dBc/Hz", range=[-150, -80]))
show(fig, 440)

### Interview questions — step 39 (9)

<details><summary><b>Q1.</b> Why is PFD/CP noise multiplied by N²?</summary>

It is referred to the PFD input like reference noise; the loop makes $\phi_{out} = N\phi_{PD}$ in band; power ×N².

</details>

<details><summary><b>Q2.</b> What is the normalised in-band phase-noise FoM?</summary>

$\mathcal L_{1Hz} = \mathcal L_{in-band} - 10\log f_{REF} - 20\log N$; good CP-PLLs reach about −220 to −230 dBc/Hz.

</details>

<details><summary><b>Q3.</b> At fixed f_out, how does in-band noise change with f_REF?</summary>

$\mathcal L = \mathcal L_{1Hz} + 10\log f_0 + 10\log N$: halving $f_{REF}$ (doubling N) costs 3 dB.

</details>

<details><summary><b>Q4.</b> Compute the in-band floor for L_1Hz = −218, f_REF = 50 MHz, N = 32.</summary>

−218 + 77.0 + 30.1 = −110.9 dBc/Hz.

</details>

<details><summary><b>Q5.</b> Where does divider noise come from?</summary>

Thermal noise and supply noise modulating the flip-flop delays; it appears as jitter on the divided edge.

</details>

<details><summary><b>Q6.</b> How does a sub-sampling PD avoid N²?</summary>

It compares the VCO directly with the reference (sampling), no divider; its noise is referred to the VCO phase directly (gain A per VCO radian), so it is not multiplied by N. Reference noise is still multiplied by N².

</details>

<details><summary><b>Q7.</b> Why is reference noise still ×N² in an SSPLL?</summary>

The loop still forces $\phi_{out} = N\phi_{REF}$: any reference jitter is a time error that equals N times more VCO radians.

</details>

<details><summary><b>Q8.</b> How much power does the divider cost in a CP-PLL?</summary>

The first stages switch at $f_{VCO}$; dynamic power $\sim CV^2f$ per stage; often the second-largest power after the VCO in ring PLLs.

</details>

<details><summary><b>Q9.</b> Is there a jitter-power trade-off in the CP?</summary>

Yes: CP current noise $\propto\sqrt{I}$ while gain $\propto I$, so SNR improves with $\sqrt{I_p}$; more current = lower normalised floor, more power and area.

</details>

<a id='step-40'></a>
## Step 40 — Limit: CP mismatch, leakage, charge sharing, clock feedthrough → ripple → reference spurs

*Roadmap step 40: spur derivation*

### What happens
At lock an ideal CP delivers no net charge and $V_{cont}$ is perfectly still. Real charge pumps disturb $V_{cont}$ once per reference
period, the VCO is frequency-modulated at $f_{REF}$ and **reference spurs** appear at $f_{out}\pm f_{REF}$.

### How (one mechanism at a time)
- **UP/DN mismatch** $\Delta I$ with reset pulses $t_{on}$: the net charge $\Delta I\,t_{on}$ must be cancelled, so the loop settles at a
  **static phase offset** $\Delta\phi = 2\pi\frac{\Delta I\,t_{on}}{I_pT_{REF}}$; each period a short positive and negative current
  pulse remain → ripple.
- **Leakage** $I_{leak}$ (CP off-current, MOS-cap or varactor leakage) discharges the filter between pulses; the loop answers with a
  pulse of charge $I_{leak}T_{REF}$ each period → offset $2\pi I_{leak}/I_p$ and a sawtooth ripple.
- **Charge sharing** (internal CP nodes at the rail when off, step 42), **charge injection and clock feedthrough** of the switches.

**Spur level** (narrowband FM, first harmonic $V_1$ of the ripple):
$$\text{spur} = 20\log_{10}\frac{K_{VCO}|V_1|}{2\omega_{REF}},\qquad V_1 = |c_1|\cdot|Z(j\omega_{REF})|,$$
with $c_1$ the first Fourier coefficient of the CP current. The code below does this calculation for the step-38 design.

**What the numbers show.** With ideal switches, mismatch and leakage alone give very low spurs for this filter (mismatch makes a
*doublet* — two opposite charges 150 ps apart — that almost cancels at 50 MHz). Charge injection and clock feedthrough of real
switches (tens of fC, not cancelled within the period) dominate: compare the printed injection results.

### Why it matters
The proposal limits the spur to −45 dBc (max). The filter attenuates $f_{REF}$ only by $|Z(j\omega_{REF})|$ — a wider loop (smaller
capacitors) means larger $|Z|$ and larger spurs.

In [ ]:
def cp_spur(Ip, mism, t_on, Ileak, d, Kv, fref, nh=60, qinj=0.0):
    """Reference spur of a CP-PLL from UP/DN mismatch (fraction) and leakage (A).
    Builds one period of the charge-pump current at lock (net charge = 0), takes its Fourier series,
    passes it through the filter Z(jk w_ref) and converts the first-harmonic ripple to a spur:
    spur = 20 log(K_VCO |V1| / (2 w_ref)) (narrowband FM). Returns spur dBc, static offset (deg), t (s), v (V), i (A)."""
    T = 1/fref; Iu, Id = Ip*(1+mism/2), Ip*(1-mism/2)
    q = -(Iu - Id)*t_on + Ileak*T - qinj                    # net charge the extra pulse must supply
    ex_up = q/Iu if q > 0 else 0.0; ex_dn = -q/Id if q < 0 else 0.0
    pulses = [(0.0, t_on + ex_up, Iu), (0.0, t_on + ex_dn, -Id), (0.0, T, -Ileak),   # (start, end, current)
              (t_on + 0.3e-9, t_on + 0.32e-9, qinj/20e-12)]                          # switch charge injection at turn-off
    w0 = 2*np.pi*fref
    Z = lambda s: (1+s*d['R']*d['C1'])/(s*(d['C1']+d['C2'])*(1+s*d['R']*d['C1']*d['C2']/(d['C1']+d['C2'])))
    t = np.linspace(0, T, 4000, endpoint=False); v = np.zeros_like(t); V1 = 0
    for k in range(1, nh+1):                                 # exact Fourier coefficients of rectangular pulses
        ck = sum(I*(np.exp(-1j*k*w0*a) - np.exp(-1j*k*w0*b_))/(1j*k*w0) for a, b_, I in pulses)*2/T
        Vk = ck*Z(1j*k*w0)
        v += np.real(Vk*np.exp(1j*k*w0*t))
        if k == 1: V1 = abs(Vk)
    spur = 20*np.log10(max(Kv*V1/(2*w0), 1e-12))
    off = 360*(ex_up - ex_dn)/T
    td = np.linspace(0, 1e-9, 2001)                          # fine grid for drawing the current
    i = sum(I*((td >= a) & (td < b_)) for a, b_, I in pulses)
    t = (t, td)
    return spur, off, t, v, i

Ip, Kv, N, fref = 100e-6, 2*np.pi*300e6, 32, 50e6
d = sl.loop_design_type2(2.5e6, 60, Ip, Kv, N)
def frame(m_pct):
    sp, off, (t, td), v, i = cp_spur(Ip, m_pct/100, 150e-12, 0.0, d, Kv, fref)
    return ([dict(x=td*1e9, y=i*1e6, name="CP current (uA), first 1 ns", row=1, col=1),
             dict(x=t*1e9, y=(v - v.mean())*1e6, name="V_cont ripple (uV), whole period", row=2, col=1)],
            f"UP/DN mismatch {m_pct}% with 150 ps reset pulses: static offset {off:.3f} deg, reference spur {sp:.1f} dBc")
fig = sweep([0.5, 1, 2, 5, 10, 20], frame, "current mismatch (%)", active=2, rows=2, cols=1, shared_xaxes=True,
            subplot_titles=("charge-pump current (first 1 ns of the period)", "ripple on V_cont over one period"), height=560, maxpts=2001,
            layout=lambda f: f.update_xaxes(title_text="time within the period (ns)", row=2, col=1))
show(fig)
Il = np.logspace(-12, -7, 40)
sp = [cp_spur(Ip, 0.0, 150e-12, x, d, Kv, fref)[0] for x in Il]
fig = go.Figure(go.Scatter(x=Il, y=sp, name="spur vs leakage"))
fig.update_xaxes(type="log", title="leakage current (A)").update_yaxes(title="reference spur (dBc)")
show(fig, 360)
for q in (1e-15, 3e-15, 10e-15):
    print(f"charge injection {q*1e15:.0f} fC per period (switch turn-off, 0.3 ns after the pulse): spur {cp_spur(Ip, 0.0, 150e-12, 0.0, d, Kv, fref, qinj=q)[0]:.1f} dBc")

### Interview questions — step 40 (11)

<details><summary><b>Q1.</b> Why does CP mismatch cause a static phase offset?</summary>

With reset pulses both sources are on each cycle; the mismatch leaves net charge $\Delta I t_{on}$, so the loop shifts the edges until the longer pulse of one source supplies the opposite charge: $\Delta\phi = 2\pi\Delta I t_{on}/(I_pT)$.

</details>

<details><summary><b>Q2.</b> Why does mismatch cause spurs even though the average current is zero?</summary>

The charge is delivered as a short positive and negative pulse at slightly different times; the waveform repeats every period and has energy at $f_{REF}$ and harmonics.

</details>

<details><summary><b>Q3.</b> Derive the leakage-induced phase offset.</summary>

Charge balance: $I_p\frac{\Delta\phi}{2\pi}T = I_{leak}T$ → $\Delta\phi = 2\pi I_{leak}/I_p$.

</details>

<details><summary><b>Q4.</b> How do you estimate the spur from a ripple voltage?</summary>

First-harmonic amplitude $V_1$ of $V_{cont}$; phase modulation index $\beta = K_{VCO}V_1/\omega_{REF}$; spur $=20\log(\beta/2)$ dBc.

</details>

<details><summary><b>Q5.</b> Why are spurs worse with a wider loop?</summary>

Smaller capacitors → larger filter impedance at $f_{REF}$ → larger ripple for the same disturbance charge; also less attenuation by C2.

</details>

<details><summary><b>Q6.</b> Which leakage sources exist in a SKY130 loop filter?</summary>

CP switch off-current, junction leakage of the CP output, gate leakage of a thin-oxide MOS cap (C1), and at high temperature all of these grow exponentially.

</details>

<details><summary><b>Q7.</b> What are charge injection and clock feedthrough?</summary>

When a switch turns off, its channel charge splits to both terminals (injection); the gate edge couples through $C_{gd}$ (feedthrough). Both put charge on $V_{cont}$ each period.

</details>

<details><summary><b>Q8.</b> Why do spurs fall at f_REF ± ... and how are they measured?</summary>

The disturbance is periodic at $f_{REF}$, so it phase-modulates the VCO at $f_{REF}$; measured with a spectrum analyser as dBc relative to the carrier.

</details>

<details><summary><b>Q9.</b> Proposal spur limit and our measurement trick?</summary>

−45 dBc max (FFNC off). Measured at /16 output (+24.1 dB to refer to 1.6 GHz), and with pll_en off to separate reference feedthrough.

</details>

<details><summary><b>Q10.</b> How does the SSPLL change the spur mechanisms?</summary>

The CP is a pulsed transconductor driven by the held sample; at lock the sample is near zero so the CP current is near zero; spurs come mainly from sampler kickback and pulse feedthrough (Phase 4).

</details>

<details><summary><b>Q11.</b> What is the effect of mismatch on in-band noise?</summary>

Indirect: the static offset makes the CP operate with both sources on longer, injecting more noise; mismatch itself is a deterministic spur source.

</details>

<a id='step-41'></a>
## Step 41 — Limit summary: dead zone, C1 area, and the bandwidth–spur–noise trade-off

*Roadmap step 41: summary table*

### What happens
Changing the loop bandwidth moves four things at once: VCO-noise suppression, ×N² in-band noise, reference spurs, and filter area (plus the dead zone handled by the reset delay).

### How it happens
Wider f_c: the high-pass 1/(1+G) suppresses more VCO noise, the low-pass NG/(1+G) passes more in-band noise, smaller capacitors (C_tot ∝ 1/f_c²) raise |Z(jω_REF)| and therefore the spur, and the sampling delay costs 180°·f_c/f_REF of PM.

### Why it happens
Every term pulls the optimum in a different direction, so there is no free lunch: a ring-VCO CP-PLL ends up limited by its in-band floor and spurs, which is the case for the SSPLL.

### How it happens
For each loop bandwidth the filter is redesigned at 60° PM; the area comes from the capacitor densities, the
spur from the Fourier series of the charge-pump current through $Z(j\omega_{REF})$ (step 40), and the jitter from the phase-domain
noise model (step 23).

### Why it happens
All three depend on the same capacitors: smaller capacitors (wider loop) mean less area but a larger
impedance at $f_{REF}$ (spurs) and a wider window for ×N² noise; larger ones do the opposite. The loop bandwidth is the single knob
that moves all of them, so it is always a compromise.

### What happens
Every CP-PLL design is a three-way trade-off with the loop bandwidth in the middle:

| Raise $f_c$ → | Effect | Why |
|---|---|---|
| VCO noise | ↓ suppressed more | $1/(1+G)$ high-pass moves up |
| in-band ×N² noise | ↑ passes more | $NG/(1+G)$ low-pass widens |
| reference spur | ↑ | smaller C → larger $|Z(j\omega_{REF})|$ |
| area | ↓ | $C_{tot}\propto1/f_c^2$ |
| stability | ↓ | sampling delay $180°f_c/f_{REF}$ |

Plus the **dead zone** (fixed with reset delay, which then causes the mismatch spur). The plot computes the first three rows for the
step-38 design family (60° PM, 2% mismatch with 150 ps reset pulses).

In [ ]:
def cp_spur(Ip, mism, t_on, Ileak, d, Kv, fref, nh=60, qinj=0.0):
    """Reference spur of a CP-PLL from UP/DN mismatch (fraction) and leakage (A).
    Builds one period of the charge-pump current at lock (net charge = 0), takes its Fourier series,
    passes it through the filter Z(jk w_ref) and converts the first-harmonic ripple to a spur:
    spur = 20 log(K_VCO |V1| / (2 w_ref)) (narrowband FM). Returns spur dBc, static offset (deg), t (s), v (V), i (A)."""
    T = 1/fref; Iu, Id = Ip*(1+mism/2), Ip*(1-mism/2)
    q = -(Iu - Id)*t_on + Ileak*T - qinj                    # net charge the extra pulse must supply
    ex_up = q/Iu if q > 0 else 0.0; ex_dn = -q/Id if q < 0 else 0.0
    pulses = [(0.0, t_on + ex_up, Iu), (0.0, t_on + ex_dn, -Id), (0.0, T, -Ileak),   # (start, end, current)
              (t_on + 0.3e-9, t_on + 0.32e-9, qinj/20e-12)]                          # switch charge injection at turn-off
    w0 = 2*np.pi*fref
    Z = lambda s: (1+s*d['R']*d['C1'])/(s*(d['C1']+d['C2'])*(1+s*d['R']*d['C1']*d['C2']/(d['C1']+d['C2'])))
    t = np.linspace(0, T, 4000, endpoint=False); v = np.zeros_like(t); V1 = 0
    for k in range(1, nh+1):                                 # exact Fourier coefficients of rectangular pulses
        ck = sum(I*(np.exp(-1j*k*w0*a) - np.exp(-1j*k*w0*b_))/(1j*k*w0) for a, b_, I in pulses)*2/T
        Vk = ck*Z(1j*k*w0)
        v += np.real(Vk*np.exp(1j*k*w0*t))
        if k == 1: V1 = abs(Vk)
    spur = 20*np.log10(max(Kv*V1/(2*w0), 1e-12))
    off = 360*(ex_up - ex_dn)/T
    td = np.linspace(0, 1e-9, 2001)                          # fine grid for drawing the current
    i = sum(I*((td >= a) & (td < b_)) for a, b_, I in pulses)
    t = (t, td)
    return spur, off, t, v, i

import sys; sys.path.insert(0, "../models")
import pllmodel as pm
Ip, Kv, N, fref = 100e-6, 2*np.pi*300e6, 32, 50e6
fcs = np.logspace(np.log10(0.3e6), np.log10(5e6), 15)
area, spur, jit = [], [], []
for fc in fcs:
    d = sl.loop_design_type2(fc, 60, Ip, Kv, N)
    area.append((d['C1']/7.48e-15 + d['C2']/2.03e-15)/1e3)
    spur.append(cp_spur(Ip, 0.02, 150e-12, 0, d, Kv, fref)[0])
    jit.append(pm.jitter(pm.TYPE2_TYP.but(fc=fc), "type2")[0]["TOTAL"]*1e12)
fig = make_subplots(rows=1, cols=3, subplot_titles=("filter area (1000 um^2)", "reference spur (dBc)", "rms jitter (ps)"))
fig.add_trace(go.Scatter(x=fcs, y=area, name="area"), 1, 1)
fig.add_trace(go.Scatter(x=fcs, y=spur, name="spur"), 1, 2)
fig.add_trace(go.Scatter(x=fcs, y=jit, name="jitter"), 1, 3)
fig.update_xaxes(type="log", title_text="f_c (Hz)").update_yaxes(type="log", row=1, col=1)
show(fig, 400)

### Interview questions — step 41 (8)

<details><summary><b>Q1.</b> Summarise the CP-PLL bandwidth trade-off.</summary>

Wider: less VCO noise, smaller filter, faster lock; but more in-band (×N²) noise, larger reference spurs and less stability margin.

</details>

<details><summary><b>Q2.</b> Why does the filter area scale as 1/f_c²?</summary>

$C_{tot} = bI_pK_{VCO}/(2\pi N\omega_c^2)$.

</details>

<details><summary><b>Q3.</b> Which PLL types prefer a narrow bandwidth?</summary>

PLLs with a clean VCO (LC) and noisy reference, or jitter-cleaning PLLs that must filter an input clock.

</details>

<details><summary><b>Q4.</b> Why can't a ring-oscillator CP-PLL simply use a 10 MHz bandwidth at 50 MHz reference?</summary>

$f_{REF}/5$ is beyond the sampled-loop stability limit; the ×N² in-band floor and spurs also rise.

</details>

<details><summary><b>Q5.</b> What is the largest area block of a typical integrated CP-PLL?</summary>

The loop-filter capacitor C1 (Nagam notes the zero capacitor is typically the largest component).

</details>

<details><summary><b>Q6.</b> How can you shrink C1 without changing dynamics?</summary>

Capacitor multiplication, dual-path filters (separate integral and proportional paths with scaled currents), lower $I_p$ with a larger R1 (noise penalty), or (with FFNC) accept more R1 noise because FFNC cancels it (Phase 6).

</details>

<details><summary><b>Q7.</b> Is the dead zone still an issue in modern PLLs?</summary>

Not with a reset-delay PFD; the cost moved to mismatch-induced spurs.

</details>

<details><summary><b>Q8.</b> How do spurs and jitter combine in an rms jitter number?</summary>

Deterministic spurs add jitter $\sigma = \sqrt2\cdot10^{spur/20}/(2\pi f_0)$ per spur pair (if inside the integration band); −55 dBc at 50 MHz offset is outside 40 MHz, so it does not enter our jitter band.

</details>

<a id='step-42'></a>
## Step 42 — Fixes: reset delay, matched CP with op-amp equalised drains, dump-node buffer

*Roadmap step 42: schematics in words*

### What happens
Three circuit fixes remove the CP-PLL's own error sources: the PFD dead zone, UP/DN current mismatch over V_cont, and charge sharing when switches close.

### How it happens
Reset delay keeps both outputs on for $t_{rst} > t_{on}$; a replica branch with an op-amp forces the replica drain to $V_{cont}$ and servo-matches UP and DN (mismatch ÷ (1 + A)); current steering into a dump node held at $V_{cont}$ by a unity-gain buffer keeps internal nodes from drifting to the rails.

### Why it happens
Each error is periodic at f_REF, so each one becomes a reference spur or a static phase offset; fixing them at the source is cheaper than filtering them later with a narrower loop.

### What happens
Three circuit fixes remove the non-idealities of steps 33 and 40: the dead zone, the UP/DN current mismatch
across the control-voltage range, and the charge-sharing glitch when a charge-pump switch closes.

### How it happens
(1) Delay in the PFD reset path keeps both switches on for $t_{rst}>t_{on}$ every cycle. (2) A replica branch
and an op-amp force the replica drain to $V_{cont}$ and servo one current source until UP = DN; the output branch copies the bias, so
mismatch falls by the op-amp gain. (3) Current is steered to a dump node held at $V_{cont}$ by a unity-gain buffer instead of being
switched off, so the internal node never drifts to a rail.

### Why it happens
Each problem has a physical cause the fix removes: switches need time to turn on (dead zone); channel-length
modulation makes currents depend on drain voltage (mismatch); a floating internal node charges to a rail and shares that charge on
reconnection (glitch).

### Reset delay (dead zone)
Insert delay in the PFD reset path so UP and DN overlap for $t_{rst} > t_{on}$ of the CP switches (step 33 figure).

### Matched charge pump with op-amp equalised drains
**What:** UP (PMOS source) and DN (NMOS sink) currents depend on their drain voltages through channel-length modulation, so they
match only at one $V_{cont}$. **How:** a replica branch copies the output voltage onto the replica drains and an op-amp adjusts the
PMOS (or NMOS) bias until the replica currents are equal; the main branch then matches at any $V_{cont}$ within compliance.
Residual mismatch ≈ (unaided mismatch)/(1 + A_op-amp).

### Dump node with a unity-gain buffer (charge sharing)
**What:** when a CP switch is off, the internal node between switch and current source drifts to a rail. When the switch closes, that
node's capacitance $C_x$ shares charge with $V_{cont}$: $\Delta V = \frac{C_x(V_x-V_{cont})}{C_x+C_{filter}}$ every cycle. **How:** steer the current
to a *dump* node held at $V_{cont}$ by a unity-gain buffer instead of switching it off, so $V_x \approx V_{cont}$ always and nothing is shared.
Our SSPLL's pulsed-Gm charge pump uses the same dump-node idea (Phase 4, step 50).

In [ ]:
V = np.linspace(0.2, 1.6, 200); Vdd = 1.8
fig = go.Figure()
for lam in (0.05, 0.1, 0.2):
    up = 1 + lam*(Vdd - V - Vdd/2); dn = 1 + lam*(V - Vdd/2)            # equal at V = Vdd/2
    mis = 100*(up - dn)
    fig.add_trace(go.Scatter(x=V, y=mis, name=f"simple CP, lambda = {lam} 1/V"))
    fig.add_trace(go.Scatter(x=V, y=mis/(1+100), name=f"op-amp equalised (A = 40 dB), lambda = {lam}", line=dict(dash="dash")))
fig.update_xaxes(title="V_cont (V)").update_yaxes(title="UP - DN mismatch (%)")
show(fig, 420)
Cx, Cf = 20e-15, 1e-12
print("charge-sharing step for C_x = 20 fF at 0.9 V difference into 1 pF:", f"{Cx*0.9/(Cx+Cf)*1e3:.1f} mV per cycle")

### Interview questions — step 42 (8)

<details><summary><b>Q1.</b> Why do UP and DN currents mismatch as V_cont changes?</summary>

Channel-length modulation: the PMOS current falls as $V_{cont}$ rises (less $V_{SD}$) and the NMOS current rises; they cross at one voltage only.

</details>

<details><summary><b>Q2.</b> How does an op-amp improve CP matching?</summary>

In a replica branch it forces the replica drain to $V_{cont}$ and servo-adjusts one current source until replica UP = DN; the output branch shares the bias and drain voltage, so mismatch shrinks by the op-amp gain.

</details>

<details><summary><b>Q3.</b> What is charge sharing in a CP and how is it fixed?</summary>

Internal nodes at a rail connect to the output when a switch closes and share charge. Fix: keep those nodes at $V_{cont}$ with a unity-gain buffer and steer current to a dump node instead of turning it off.

</details>

<details><summary><b>Q4.</b> What is current steering vs switching a current source on/off?</summary>

Steering keeps the source always on and switches its current between the output and a dump path: faster, no turn-on transient, no charge-sharing node drift.

</details>

<details><summary><b>Q5.</b> How does a PFD reset delay affect CP noise?</summary>

Both currents conduct for $t_{rst}$ every cycle, so their noise is injected for that time; a longer delay means more in-band CP noise.

</details>

<details><summary><b>Q6.</b> Name a spur-reduction technique that does not change the CP.</summary>

Sampled (S&H) loop filter, spur-aware loop filter poles, higher-order filtering, randomising the reference (only in fractional-N), or a smaller $K_{VCO}$.

</details>

<details><summary><b>Q7.</b> Why do differential charge pumps help?</summary>

Common-mode disturbances (injection, feedthrough, supply) cancel; they need a common-mode feedback loop and a differential VCO control input.

</details>

<details><summary><b>Q8.</b> What is the compliance range of a CP and how does it limit the VCO range?</summary>

Output range over which both current sources stay saturated; the usable $V_{cont}$ range (hence fine-tuning range) is limited to it.

</details>

<a id='step-43'></a>
## Step 43 — Fixes: sample-reset/S&H filters, calibration, and a PD gain that does not shrink with N

*Roadmap step 43: the bridge to the SSPD*

### What happens
Further fixes act on the filter (sampled loop filter), on the calibration (trim the CP), or on the detector itself (a detector whose gain does not shrink with N).

### How it happens
A sampled filter updates $V_{cont}$ only after the CP pulse has settled, so the VCO never sees the $I_pR_1$ spike; calibration measures the static offset and trims a current source; a sub-sampling detector measures VCO phase directly with gain A per VCO radian instead of $I_p/(2\pi N)$.

### Why it happens
The last idea removes the root cause (the N² penalty on detector noise) instead of treating symptoms — the bridge to Phase 4.

### What happens
Instead of only improving the charge pump, these fixes change *what the VCO sees* (a sampled filter), trim the
remaining error (calibration), or change the detector itself so its gain referred to the VCO no longer shrinks with N.

### How it happens
A sampled loop filter lets the charge-pump pulse settle on a separate capacitor and transfers the result to
the VCO node only after the pulse; a calibration loop measures the static phase offset and adjusts one current source; a
sub-sampling detector samples the VCO itself so its gain is $A$ volts per VCO radian instead of $I_p/(2\pi N)$.

### Why it happens
Spurs come from the fast transient on $V_{cont}$, which the hold hides; mismatch is a fixed error that a
measurement can cancel; and the N² noise penalty comes from detecting the *divided* phase — remove the divider from the detection and
the penalty disappears.

### Sampled (S&H) loop filter
**What:** isolate the VCO from the charge-pump activity. **How:** let the CP pulses settle on a separate capacitor, then *sample* the
result onto the VCO control node after the pulse has ended. $V_{cont}$ changes once per period by a clean step, with no $I_pR_1$ spike
in between. **Why:** the spur energy comes from the spike; a held voltage contains only the slow loop correction.

### Calibration
Measure and trim CP mismatch (e.g. adjust one current source until the static phase offset is zero), or trim $K_{VCO}$/bandwidth.

### A PD whose gain does not shrink with N
In a CP-PLL, the detector gain *referred to the VCO* is $\frac{I_p}{2\pi N}$ per VCO radian: a larger N makes the detector "see" less of
the VCO's phase error, so its own noise matters N² more. A **sub-sampling PD** measures the VCO phase directly, with gain
$K = A$ (V per VCO radian) independent of N — Phase 4.

In [ ]:
T = 20e-9; Ip = 100e-6; R1, C1, C2 = 18.05e3, 13.16e-12, 1.018e-12; tw = 0.5e-9
t = np.linspace(0, 3*T, 3000); dt = t[1]-t[0]
v1 = v2 = 0.0; out = []
for tt in t:
    i = Ip if (tt % T) < tw else 0.0
    i_r = (v2 - v1)/R1
    v1 += i_r/C1*dt; v2 += (i - i_r)/C2*dt
    out.append(v2)
out = np.array(out)
ts = np.floor((t - 0.5*T)/T)*T + 0.5*T                       # last sampling instant (mid-period, after the pulse)
held = np.where(t >= 0.5*T, out[np.clip(np.round(ts/dt).astype(int), 0, len(t)-1)], 0.0)
fig = go.Figure([go.Scatter(x=t*1e9, y=out*1e3, name="V_cont, plain filter (spike I_p R1 each pulse)"),
                 go.Scatter(x=t*1e9, y=held*1e3, name="sampled filter output (updated once per period)", line=dict(shape="hv"))])
fig.update_xaxes(title="time (ns)").update_yaxes(title="mV")
show(fig, 400)

### Interview questions — step 43 (8)

<details><summary><b>Q1.</b> How does a sampled loop filter reduce spurs?</summary>

It hides the CP transient from the VCO: the VCO sees only a held voltage updated after the pulse settles, removing the $I_pR_1$ spike and much of the switching glitch.

</details>

<details><summary><b>Q2.</b> What does a sampled filter cost?</summary>

Extra delay (phase margin), extra switch charge injection, and complexity.

</details>

<details><summary><b>Q3.</b> What is the CP gain referred to VCO phase?</summary>

$I_p/(2\pi N)$ A per VCO radian — it shrinks with N, so the detector's own noise is relatively N² larger.

</details>

<details><summary><b>Q4.</b> What is the sub-sampling PD gain referred to VCO phase?</summary>

$A$ V per VCO radian (sampled amplitude), independent of N.

</details>

<details><summary><b>Q5.</b> How can you calibrate CP mismatch in the background?</summary>

Measure the static phase offset or the average error (e.g. with a TDC or a replica) and adjust one current source digitally until the offset is zero.

</details>

<details><summary><b>Q6.</b> Why does the SSPLL need less spur filtering than a CP-PLL?</summary>

At lock the sampled error is ~0, so the pulsed Gm injects almost no charge; there is no PFD reset pulse with both currents on.

</details>

<details><summary><b>Q7.</b> What is the 'reference spur' mechanism unique to sampling PDs?</summary>

Kickback: the sampling switch disturbs the VCO node when it opens/closes and loads it with a time-varying capacitance.

</details>

<details><summary><b>Q8.</b> Name a modern PD that keeps the PFD's acquisition but adds sub-sampling's low noise.</summary>

Hsu et al.'s PFD with an auxiliary sub-sampling PD (CICC 2011), and the sub-sampling-assisted PFD papers in the project files.

</details>

<a id='step-44'></a>
## Step 44 — Fix for ring-VCO noise: a wider f_c needs lower in-band noise

*Roadmap step 44: bridge note: motivates the SSPLL*

### What happens
For a ring VCO the jitter-optimal bandwidth slides to wider values and lower jitter as the in-band noise floor drops.

### How it happens
The total jitter is the integral of in-band noise × |NG/(1+G)|² plus VCO noise × |1/(1+G)|²; lowering the in-band term lets the crossover move up until the stability/spur limit (~f_REF/20 for an SSPLL) is reached.

### Why it happens
A CP-PLL can only lower its floor with more current or a higher f_REF; the SSPLL removes ~20 dB at once, and beyond the bandwidth wall only feedforward cancellation can help.

### How it happens
For each in-band floor the model sweeps the loop bandwidth and integrates the output phase noise
(10 kHz–40 MHz); the minimum of the jitter curve is the optimum bandwidth, and the dotted line marks the $f_{REF}/20$ limit set by
sampling delay and spurs.

### What happens
With a ring VCO, the jitter-optimal bandwidth depends on the in-band floor: the lower the floor, the wider the optimum, and the lower the
jitter. Move the slider: the curve's minimum slides right and down as the floor drops from CP-PLL levels (−111 dBc/Hz) to SSPLL levels
(−130 dBc/Hz) — until it hits the $f_{REF}/20$ stability/spur wall.

### Why this is the case for an SSPLL
A CP-PLL can only lower its floor by more CP current (power, area) or a higher $f_{REF}$ (crystal limits). The SSPLL removes the N²
factor from the detector noise — an instant ~20 dB — so the loop can use the full $f_{REF}/20$ bandwidth productively. Beyond that wall,
the only remaining lever on ring noise is **feedforward cancellation** (Phase 6).

In [ ]:
import sys; sys.path.insert(0, "../models")
import pllmodel as pm
fcs = np.logspace(5, np.log10(6.25e6), 40)
def frame(floor):
    d = pm.TYP.but(pd=floor)
    j = np.array([pm.jitter(d.but(fc=fc), "sspll")[0]["TOTAL"] for fc in fcs])*1e12
    i = np.argmin(j)
    return ([dict(x=fcs, y=j, name="rms jitter vs bandwidth"),
             dict(x=[2.5e6, 2.5e6], y=[0, 20], name="f_REF/20 wall", line=dict(dash="dot", color="grey"))],
            f"in-band floor {floor} dBc/Hz: optimum f_c = {fcs[i]/1e6:.2f} MHz -> {j[i]:.2f} ps; at 2.5 MHz {np.interp(2.5e6, fcs, j):.2f} ps")
fig = sweep([-105, -111, -115, -120, -125, -130, -135], frame, "in-band floor at the output (dBc/Hz)", active=1,
            layout=lambda f: f.update_xaxes(type="log", title="loop bandwidth f_c (Hz)").update_yaxes(title="ps", range=[0, 10]))
show(fig, 440)

### Interview questions — step 44 (6)

<details><summary><b>Q1.</b> Why does a ring-VCO PLL benefit more from low in-band noise than an LC PLL?</summary>

The ring's free-running noise is high, so the jitter-optimal bandwidth is wide; at a wide bandwidth the in-band floor sets the total.

</details>

<details><summary><b>Q2.</b> What two levers lower a CP-PLL's in-band floor?</summary>

More CP current (lower normalised floor) and a higher reference frequency (lower N).

</details>

<details><summary><b>Q3.</b> By how much does an SSPLL lower the detector floor compared with a CP-PLL here?</summary>

From about −111 to −130 dBc/Hz output-referred in our budget (~19 dB), mostly by removing the N² factor.

</details>

<details><summary><b>Q4.</b> What limits the SSPLL bandwidth once its floor is low?</summary>

Sampling and hold delay (stability) and spur/ripple considerations: ~$f_{REF}/20$; the reference floor ×N² also remains.

</details>

<details><summary><b>Q5.</b> After the bandwidth wall, what else can reduce ring-VCO jitter?</summary>

More VCO power, better VCO topology, supply regulation, or feedforward noise cancellation (FFNC) which works beyond the loop bandwidth up to ~$f_{REF}/8$.

</details>

<details><summary><b>Q6.</b> Why is the optimum bandwidth of the SSPLL in this model at the f_REF/8 edge?</summary>

The model has no sampling-delay penalty; the real loop is limited near $f_{REF}/20$ — treat model results beyond that as unreliable.

</details>

<a id='step-45'></a>
## Step 45 — Lab: behavioural Type-II PLL at 1.6 GHz (frequency steps, slewing, settling)

*Roadmap step 45: results*

### What happens
A behavioural CP-PLL is hit with a 200 MHz output-frequency step: a wide loop tracks it linearly, a narrow loop saturates the PFD and slews.

### How it happens
The cycle-by-cycle simulator computes the PFD error each reference period; when |Δφ| ≥ 2π the PFD stays in one state and the CP delivers its maximum average current, so $V_{cont}$ ramps at ≈ $I_p/C_{tot}$ until the error is back inside the linear range.

### Why it happens
Lock time and acquisition behaviour depend on loop bandwidth and frequency error, not only on the linear step response — why the lock-time spec starts with the coarse code already set.

### How it happens
The same cycle-by-cycle simulator as step 24 applies a 200 MHz output-frequency step as a phase ramp at the
reference; the PFD error is clipped at ±2π, so when the loop cannot follow, the charge pump delivers its maximum average current and
$V_{cont}$ ramps.

### Why it happens
A wide loop corrects the phase before it grows beyond the PFD range; a narrow loop cannot, the phase error
exceeds 2π, the PFD saturates and the loop can only move the VCO frequency as fast as $I_p/C_{tot}$ allows — slew-limited acquisition.

### What you do
Use the cycle-by-cycle simulator with the step-38 filter ($I_p$ = 100 µA, $R_1$ = 18.05 kΩ, $C_1$ = 13.16 pF, $C_2$ = 1.02 pF,
$K_{VCO}$ = 2π·300 MHz/V, N = 32), apply a 200 MHz output-frequency step, and vary the loop bandwidth (the filter is redesigned
for each $f_c$ at 60° PM). A wide loop tracks the step linearly. A narrow loop cannot follow: the phase error exceeds the PFD's ±2π
range, the PFD saturates, the CP delivers its maximum average current and $V_{cont}$ **slews** — this is how a PFD acquires frequency.
The final $V_{cont}$ must equal $\Delta\omega/K_{VCO}$ = 667 mV.

**Transistor-level baseline:** the optional SKY130 Type-II implementation (same VCO, CP and filter as computed here) is step 88 of
Phase 7; the numbers here are its targets.

In [ ]:
Ip, Kv, N, fref = 100e-6, 2*np.pi*300e6, 32, 50e6
d = sl.loop_design_type2(2.5e6, 60, Ip, Kv, N)
def frame(fc_MHz):
    d = sl.loop_design_type2(fc_MHz*1e6, 60, Ip, Kv, N)
    t, e, vc = sl.cp_pll_sim(Ip, d['R'], d['C1'], d['C2'], Kv, N, fref, ncyc=2000, f_step=200e6/N)
    sat = np.mean(abs(e) >= 2*np.pi)
    return ([dict(x=t*1e6, y=vc*1e3, name="V_cont (mV)", row=1, col=1),
             dict(x=[0, t[-1]*1e6], y=[2*np.pi*200e6/Kv*1e3]*2, name="expected final V_cont", line=dict(dash="dot"), row=1, col=1),
             dict(x=t*1e6, y=np.clip(e, -2*np.pi, 2*np.pi), name="PFD phase error (rad, clipped at 2pi)", row=2, col=1)],
            f"200 MHz output step, f_c = {fc_MHz} MHz: PFD saturated {100*sat:.0f}% of cycles; V_cont after 40 us {vc[-1]*1e3:.0f} mV "
            f"(expected {2*np.pi*200e6/Kv*1e3:.0f} mV)")
fig = sweep([0.2, 0.3, 0.5, 1.0, 2.5], frame, "loop bandwidth f_c (MHz)", active=1, rows=2, cols=1, shared_xaxes=True,
            subplot_titles=("control voltage", "phase error at the PFD"), height=600,
            layout=lambda f: f.update_xaxes(title_text="time (us)", row=2, col=1))
show(fig)

**What to look for.** At 2.5 MHz the PFD never saturates: its ±2π range is generous for a fast loop. At 0.3 MHz it saturates for 28% of the cycles and at 0.2 MHz almost always: $V_{cont}$ ramps (slew-limited acquisition), then linear settling takes over. Lock time grows with the frequency error and with a narrower loop — the reason the proposal's lock time is specified from `pll_en` with the coarse code already set.

### Interview questions — step 45 (8)

<details><summary><b>Q1.</b> Why does a large frequency step make V_cont ramp linearly?</summary>

The phase error exceeds the PFD range, the PFD stays in one state, the CP delivers ~constant maximum current into the filter: a constant slope $I_p/C_{tot}$.

</details>

<details><summary><b>Q2.</b> What is the slew rate of V_cont during acquisition here?</summary>

At full PFD duty the CP current goes into the capacitors: $I_p/(C_1+C_2) \approx 100\,\mu A/14.2\,pF \approx 7$ V/µs; times $K_{VCO}$ this is how fast the VCO frequency can move (here ~2 GHz/µs, so the average over the slewing phase is set by the duty).

</details>

<details><summary><b>Q3.</b> How do you reduce lock time for large steps?</summary>

Coarse digital calibration of the VCO band, bandwidth boosting during acquisition, or a faster frequency detector path.

</details>

<details><summary><b>Q4.</b> How do you verify a behavioural model against a transistor-level PLL?</summary>

Match block gains ($K_{VCO}$, $I_p$, filter) from transistor sims, then compare lock transients, final $V_{cont}$, overshoot and settling time.

</details>

<details><summary><b>Q5.</b> What does the final V_cont tell you?</summary>

The VCO gain: $\Delta V = \Delta\omega/K_{VCO}$; any discrepancy reveals a different $K_{VCO}$ (or a non-linear tuning curve) in the circuit.

</details>

<details><summary><b>Q6.</b> Why do lock-time specs start 'with the coarse code set'?</summary>

Coarse calibration removes the large frequency error; the remaining lock is fast and predictable. Including coarse search would make lock time depend on the search algorithm.

</details>

<details><summary><b>Q7.</b> What would a cycle slip look like in this simulation?</summary>

The phase error crossing ±2π (wrapping) during the transient instead of decaying monotonically.

</details>

<details><summary><b>Q8.</b> What do you expect from the SKY130 Type-II baseline compared with this model?</summary>

Same bandwidth and PM if gains match; extra effects: CP mismatch/leakage spurs, dead-zone residue, non-linear $K_{VCO}$, divider jitter, supply noise.

</details>